# SAM2-UNet — Conv Adapter + Dice Loss

**Cải tiến kết hợp:**

1. **Conv Adapter** — thay MLP Adapter gốc bằng Conv Adapter có spatial context
2. **Dice Loss** — thêm vào loss function (đã validate riêng lẻ)

| | Baseline | Cải tiến |
|---|---|---|
| Adapter | Linear→32→Linear (MLP) | Linear→64→DWConv→Linear (Conv) |
| Loss | wIoU + wBCE | wIoU + wBCE + 0.3×Dice |
| Decoder | Giữ nguyên | Giữ nguyên |

**Lý do Conv Adapter tốt hơn MLP Adapter:**
MLP chỉ mix thông tin theo channel. DWConv1d cho phép mỗi token interact với token lân cận → học spatial context trong encoder.

## 1. Setup

In [ ]:
!git clone https://github.com/WZH0120/SAM2-UNet.git

Cloning into 'SAM2-UNet'...
remote: Enumerating objects: 316, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 316 (delta 85), reused 58 (delta 58), pack-reused 208 (from 2)
Receiving objects: 100% (316/316), 3.26 MiB | 45.67 MiB/s, done.
Resolving deltas: 100% (125/125), done.


In [ ]:
%cd SAM2-UNet

/content/SAM2-UNet


In [ ]:
!pip install -r requirements.txt
!pip install -q gdown

Looking in links: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 5.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 129.3 MB/s eta 0:00:00
   ━━━━━━━━━━━

## 2. Checkpoint

In [ ]:
!mkdir -p /content/checkpoints
!wget -q -O /content/checkpoints/sam2_hiera_large.pt \
  https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
!ls -lh /content/checkpoints

total 857M
-rw-r--r-- 1 root root 857M Jul 28  2024 sam2_hiera_large.pt


## 3. Dataset
> Bỏ qua nếu data đã có.

In [ ]:
!mkdir -p /content/data/Polyp
%cd /content/data/Polyp
!gdown --fuzzy 'https://drive.google.com/file/d/1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb/view?usp=sharing' -O TrainDataset.zip
!gdown --fuzzy 'https://drive.google.com/file/d/1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao/view?usp=sharing' -O TestDataset.zip
!unzip -q TrainDataset.zip
!unzip -q TestDataset.zip

/content/data/Polyp
Downloading...
From (original): https://drive.google.com/uc?id=1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb
From (redirected): https://drive.google.com/uc?id=1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb&confirm=t&uuid=5896258c-1808-4c81-8d5d-e2838fa96681
To: /content/data/Polyp/TrainDataset.zip
100% 419M/419M [00:06<00:00, 69.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao
From (redirected): https://drive.google.com/uc?id=1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao&confirm=t&uuid=34438cf1-f884-4cb8-ae54-de5f76351cf9
To: /content/data/Polyp/TestDataset.zip
100% 343M/343M [00:07<00:00, 47.1MB/s]


In [ ]:
import os
from pathlib import Path
root=Path('/content/data/Polyp')
for p in ['TrainDataset/images','TrainDataset/masks',
    'TestDataset/Kvasir/images','TestDataset/Kvasir/masks',
    'TestDataset/CVC-ClinicDB/images','TestDataset/CVC-ClinicDB/masks',
    'TestDataset/CVC-ColonDB/images','TestDataset/CVC-ColonDB/masks',
    'TestDataset/CVC-300/images','TestDataset/CVC-300/masks',
    'TestDataset/ETIS-LaribPolypDB/images','TestDataset/ETIS-LaribPolypDB/masks']:
    full=root/p
    print(p,'=>',len(os.listdir(full)) if full.exists() else 'MISSING')
src='/content/data/Polyp/TrainDataset/image'
dst='/content/data/Polyp/TrainDataset/images'
if os.path.exists(src) and not os.path.exists(dst):
    os.symlink(src,dst); print('Symlink created')

TrainDataset/images => MISSING
TrainDataset/masks => 1450
TestDataset/Kvasir/images => 100
TestDataset/Kvasir/masks => 100
TestDataset/CVC-ClinicDB/images => 62
TestDataset/CVC-ClinicDB/masks => 62
TestDataset/CVC-ColonDB/images => 380
TestDataset/CVC-ColonDB/masks => 380
TestDataset/CVC-300/images => 60
TestDataset/CVC-300/masks => 60
TestDataset/ETIS-LaribPolypDB/images => 196
TestDataset/ETIS-LaribPolypDB/masks => 196
Symlink created


## 4. Ghi file vào repo

In [ ]:
%%writefile /content/SAM2-UNet/SAM2UNet_convadapter.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from sam2.build_sam import build_sam2


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)

    def forward(self, x1, x2):
        x1    = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1    = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                            diffY // 2, diffY - diffY // 2])
        return self.conv(torch.cat([x2, x1], dim=1))


class ConvAdapter(nn.Module):
    """
    Conv Adapter — cải tiến từ MLP Adapter gốc.

    Adapter gốc:  Linear(dim→32) → GELU → Linear(32→dim) → GELU
                  Chỉ mix thông tin theo channel, không có spatial context.

    Conv Adapter: Linear(dim→64) → GELU
                  → DWConv1d(64, k=3)    ← capture local token interactions
                  → Linear(64→dim) → GELU

    DWConv1d áp dụng theo chiều token sequence (B, N, C):
    - Mỗi token interact với 2 neighbor tokens (k=3)
    - Trong Hiera, tokens gần nhau = patches gần nhau trong ảnh
    - → Model học được spatial context trong adapter, không chỉ channel mixing

    Params thêm so với MLP Adapter (bottleneck=32):
    - MLP:  dim×32 + 32×dim = 2×dim×32
    - Conv: dim×64 + 64×3 + 64×dim = 2×dim×64 + 192
    - Delta: ~2×dim×32 + 192 ≈ +3.5M tổng 4 stages
    """

    def __init__(self, blk, bottleneck=64):
        super().__init__()
        self.block   = blk
        dim          = blk.attn.qkv.in_features

        self.down    = nn.Linear(dim, bottleneck, bias=False)
        self.act1    = nn.GELU()
        # Depthwise conv dọc theo sequence — capture local spatial context
        # DWConv2d vì Hiera token shape là (B, H, W, C)
        self.dw_conv = nn.Conv2d(bottleneck, bottleneck,
                                 kernel_size=3, padding=1,
                                 groups=bottleneck, bias=False)
        self.up      = nn.Linear(bottleneck, dim, bias=False)
        self.act2    = nn.GELU()

    def forward(self, x):
        # x: (B, H, W, C) — Hiera dùng spatial format, không phải sequence
        prompt = self.down(x)                        # (B, H, W, bottleneck)
        prompt = self.act1(prompt)
        prompt = prompt.permute(0, 3, 1, 2)          # (B, bottleneck, H, W)
        prompt = self.dw_conv(prompt)                # 2D spatial context
        prompt = prompt.permute(0, 2, 3, 1)          # (B, H, W, bottleneck)
        prompt = self.up(prompt)                     # (B, H, W, dim)
        prompt = self.act2(prompt)
        return self.block(x + prompt)


class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size,
                 stride=1, padding=0, dilation=1):
        super().__init__()
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size,
                              stride=stride, padding=padding,
                              dilation=dilation, bias=False)
        self.bn   = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.bn(self.conv(x))


class RFB_modified(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()
        self.relu    = nn.ReLU(True)
        self.branch0 = nn.Sequential(BasicConv2d(in_channel, out_channel, 1))
        self.branch1 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 3), padding=(0, 1)),
            BasicConv2d(out_channel, out_channel, kernel_size=(3, 1), padding=(1, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 5), padding=(0, 2)),
            BasicConv2d(out_channel, out_channel, kernel_size=(5, 1), padding=(2, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5)
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv2d(out_channel, out_channel, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7)
        )
        self.conv_cat = BasicConv2d(4 * out_channel, out_channel, 3, padding=1)
        self.conv_res = BasicConv2d(in_channel, out_channel, 1)

    def forward(self, x):
        x_cat = self.conv_cat(torch.cat(
            (self.branch0(x), self.branch1(x), self.branch2(x), self.branch3(x)), 1
        ))
        return self.relu(x_cat + self.conv_res(x))


class SAM2UNet_ConvAdapter(nn.Module):
    def __init__(self, checkpoint_path=None):
        super().__init__()

        model_cfg = "sam2_hiera_l.yaml"
        model = build_sam2(model_cfg, checkpoint_path) if checkpoint_path else build_sam2(model_cfg)

        del model.sam_mask_decoder
        del model.sam_prompt_encoder
        del model.memory_encoder
        del model.memory_attention
        del model.mask_downsample
        del model.obj_ptr_tpos_proj
        del model.obj_ptr_proj
        del model.image_encoder.neck

        self.encoder = model.image_encoder.trunk

        for param in self.encoder.parameters():
            param.requires_grad = False

        # Conv Adapter thay MLP Adapter
        self.encoder.blocks = nn.Sequential(
            *[ConvAdapter(b, bottleneck=64) for b in self.encoder.blocks]
        )

        self.rfb1 = RFB_modified(144,  64)
        self.rfb2 = RFB_modified(288,  64)
        self.rfb3 = RFB_modified(576,  64)
        self.rfb4 = RFB_modified(1152, 64)

        self.up1 = Up(128, 64)
        self.up2 = Up(128, 64)
        self.up3 = Up(128, 64)

        self.side1 = nn.Conv2d(64, 1, kernel_size=1)
        self.side2 = nn.Conv2d(64, 1, kernel_size=1)
        self.head  = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        x1, x2, x3, x4 = self.encoder(x)
        x1 = self.rfb1(x1)
        x2 = self.rfb2(x2)
        x3 = self.rfb3(x3)
        x4 = self.rfb4(x4)

        x    = self.up1(x4, x3)
        out1 = F.interpolate(self.side1(x), scale_factor=16, mode='bilinear')
        x    = self.up2(x, x2)
        out2 = F.interpolate(self.side2(x), scale_factor=8,  mode='bilinear')
        x    = self.up3(x, x1)
        out  = F.interpolate(self.head(x),  scale_factor=4,  mode='bilinear')

        return out, out1, out2


if __name__ == "__main__":
    with torch.no_grad():
        model = SAM2UNet_ConvAdapter().cuda()
        x = torch.randn(1, 3, 352, 352).cuda()
        out, out1, out2 = model(x)
        total     = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Total    : {total/1e6:.3f}M")
        print(f"Trainable: {trainable/1e6:.3f}M")
        print(f"Baseline : ~4.38M trainable")

Overwriting /content/SAM2-UNet/SAM2UNet_convadapter.py


In [ ]:
%%writefile /content/SAM2-UNet/train_convadapter.py
import os
import argparse
import torch
import torch.optim as opt
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from dataset import FullDataset
from SAM2UNet_convadapter import SAM2UNet_ConvAdapter

parser = argparse.ArgumentParser("SAM2-UNet — Conv Adapter + Dice Loss")
parser.add_argument("--hiera_path",       type=str,   required=True)
parser.add_argument("--train_image_path", type=str,   required=True)
parser.add_argument("--train_mask_path",  type=str,   required=True)
parser.add_argument("--save_path",        type=str,   required=True)
parser.add_argument("--epoch",            type=int,   default=20)
parser.add_argument("--lr",               type=float, default=0.001)
parser.add_argument("--batch_size",       type=int,   default=12)
parser.add_argument("--weight_decay",     type=float, default=5e-4)
parser.add_argument("--dice_w",           type=float, default=0.3)
args = parser.parse_args()


def structure_loss(pred, mask):
    weit  = 1 + 5 * torch.abs(
        F.avg_pool2d(mask, kernel_size=31, stride=1, padding=15) - mask
    )
    wbce  = F.binary_cross_entropy_with_logits(pred, mask, reduce='none')
    wbce  = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3))
    pred_ = torch.sigmoid(pred)
    inter = ((pred_ * mask) * weit).sum(dim=(2, 3))
    union = ((pred_ + mask) * weit).sum(dim=(2, 3))
    wiou  = 1 - (inter + 1) / (union - inter + 1)
    return (wbce + wiou).mean()


def dice_loss(pred, mask, smooth=1.0):
    pred_ = torch.sigmoid(pred).view(pred.size(0), -1)
    mask_ = mask.view(mask.size(0), -1)
    inter = (pred_ * mask_).sum(dim=1)
    return (1 - (2*inter + smooth) / (pred_.sum(dim=1) + mask_.sum(dim=1) + smooth)).mean()


def combined_loss(pred, mask, dice_w):
    return structure_loss(pred, mask) + dice_w * dice_loss(pred, mask)


def main(args):
    dataset    = FullDataset(args.train_image_path, args.train_mask_path, 352, mode='train')
    dataloader = DataLoader(dataset, batch_size=args.batch_size,
                            shuffle=True, num_workers=8)

    device = torch.device("cuda")
    model  = SAM2UNet_ConvAdapter(args.hiera_path).to(device)

    optim     = opt.AdamW([{"params": model.parameters(), "initia_lr": args.lr}],
                          lr=args.lr, weight_decay=args.weight_decay)
    scheduler = CosineAnnealingLR(optim, args.epoch, eta_min=1e-7)

    os.makedirs(args.save_path, exist_ok=True)

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total: {total/1e6:.3f}M | Trainable: {trainable/1e6:.3f}M")
    print(f"Loss: structure + {args.dice_w}×Dice")

    for epoch in range(args.epoch):
        for i, batch in enumerate(dataloader):
            x      = batch['image'].to(device)
            target = batch['label'].to(device)

            optim.zero_grad()
            out, out1, out2 = model(x)

            loss = (combined_loss(out,  target, args.dice_w) +
                    combined_loss(out1, target, args.dice_w) +
                    combined_loss(out2, target, args.dice_w))

            loss.backward()
            optim.step()

            if i % 50 == 0:
                print(f"epoch:{epoch+1}-{i+1}: loss:{loss.item():.4f}")

        scheduler.step()

        if (epoch + 1) % 5 == 0 or (epoch + 1) == args.epoch:
            ckpt = os.path.join(args.save_path, f'SAM2-UNet-{epoch+1}.pth')
            torch.save(model.state_dict(), ckpt)
            print(f'[Saved] {ckpt}')


if __name__ == "__main__":
    main(args)


Writing /content/SAM2-UNet/train_convadapter.py


In [ ]:
%%writefile /content/SAM2-UNet/test_convadapter.py
import argparse
import os
import torch
import imageio
import numpy as np
import torch.nn.functional as F
from SAM2UNet_convadapter import SAM2UNet_ConvAdapter
from dataset import TestDataset

parser = argparse.ArgumentParser()
parser.add_argument("--checkpoint",      type=str, required=True)
parser.add_argument("--test_image_path", type=str, required=True)
parser.add_argument("--test_gt_path",    type=str, required=True)
parser.add_argument("--save_path",       type=str, required=True)
args = parser.parse_args()

device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_loader = TestDataset(args.test_image_path, args.test_gt_path, 352)

model = SAM2UNet_ConvAdapter()
model.load_state_dict(torch.load(args.checkpoint), strict=True)
model.to(device)
model.eval()
model.cuda()

os.makedirs(args.save_path, exist_ok=True)

for i in range(test_loader.size):
    with torch.no_grad():
        image, gt, name = test_loader.load_data()
        gt    = np.asarray(gt, np.float32)
        image = image.to(device)

        res, _, _ = model(image)

        res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
        res = res.sigmoid().data.cpu().numpy().squeeze()
        res = (res - res.min()) / (res.max() - res.min() + 1e-8)
        res = (res * 255).astype(np.uint8)

        print("Saving " + name)
        imageio.imsave(os.path.join(args.save_path, name[:-4] + ".png"), res)


Writing /content/SAM2-UNet/test_convadapter.py


In [ ]:
%cd /content/SAM2-UNet
!ls SAM2UNet_convadapter.py train_convadapter.py test_convadapter.py

/content/SAM2-UNet
SAM2UNet_convadapter.py  test_convadapter.py  train_convadapter.py


## 5. Kiểm tra params

In [ ]:
import sys, torch
sys.path.insert(0,'/content/SAM2-UNet')
from SAM2UNet_convadapter import SAM2UNet_ConvAdapter
m=SAM2UNet_ConvAdapter('/content/checkpoints/sam2_hiera_large.pt')
total=sum(p.numel() for p in m.parameters())
tr=sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'Total    : {total/1e6:.3f}M')
print(f'Trainable: {tr/1e6:.3f}M')
print(f'Baseline : ~4.38M trainable')

Total    : 218.087M
Trainable: 5.938M
Baseline : ~4.38M trainable


## 6. Smoke test (1 epoch)

In [ ]:
%cd /content/SAM2-UNet

!python train_convadapter.py \
  --hiera_path /content/checkpoints/sam2_hiera_large.pt \
  --train_image_path /content/data/Polyp/TrainDataset/images/ \
  --train_mask_path  /content/data/Polyp/TrainDataset/masks/ \
  --save_path /content/ckpt/convadapter_smoke/ \
  --epoch 1 --batch_size 2

/content/SAM2-UNet
Total: 218.105M | Trainable: 5.956M
Loss: structure + 0.3×Dice
/usr/local/lib/python3.12/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  warnings.warn(warning.format(ret))
epoch:1-1: loss:5.4760
epoch:1-51: loss:2.8210
epoch:1-101: loss:1.6393
epoch:1-151: loss:2.2762
epoch:1-201: loss:1.5959
epoch:1-251: loss:4.7573
epoch:1-301: loss:2.0663
epoch:1-351: loss:1.3091
epoch:1-401: loss:1.0568
epoch:1-451: loss:3.6707
epoch:1-501: loss:0.7481
epoch:1-551: loss:1.0068
epoch:1-601: loss:3.1804
epoch:1-651: loss:1.7794
epoch:1-701: loss:2.4612
[Saved] /content/ckpt/convadapter_smoke/SAM2-UNet-1.pth


In [ ]:
%cd /content/SAM2-UNet

!python train_convadapter.py \
  --hiera_path /content/checkpoints/sam2_hiera_large.pt \
  --train_image_path /content/data/Polyp/TrainDataset/images/ \
  --train_mask_path  /content/data/Polyp/TrainDataset/masks/ \
  --save_path /content/ckpt/convadapter_lr3e4/ \
  --epoch 1 --batch_size 2 \
  --lr 3e-4

/content/SAM2-UNet
Total: 218.105M | Trainable: 5.956M
Loss: structure + 0.3×Dice
/usr/local/lib/python3.12/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  warnings.warn(warning.format(ret))
epoch:1-1: loss:5.3207
epoch:1-51: loss:2.8787
epoch:1-101: loss:3.2514
epoch:1-151: loss:2.9719
epoch:1-201: loss:1.5527
epoch:1-251: loss:1.3053
epoch:1-301: loss:2.4339
epoch:1-351: loss:1.5440
epoch:1-401: loss:2.5649
epoch:1-451: loss:3.0526
epoch:1-501: loss:2.1787
epoch:1-551: loss:1.9129
epoch:1-601: loss:0.9006
epoch:1-651: loss:1.4464
epoch:1-701: loss:0.7667
[Saved] /content/ckpt/convadapter_lr3e4/SAM2-UNet-1.pth


## 7. Full training (20 epochs)

In [ ]:
%cd /content/SAM2-UNet

!python train_convadapter.py \
  --hiera_path /content/checkpoints/sam2_hiera_large.pt \
  --train_image_path /content/data/Polyp/TrainDataset/images/ \
  --train_mask_path  /content/data/Polyp/TrainDataset/masks/ \
  --save_path /content/ckpt/convadapter/ \
  --epoch 20 --batch_size 12 --lr 3e-4 --dice_w 0.3

/content/SAM2-UNet
Total: 218.105M | Trainable: 5.956M
Loss: structure + 0.3×Dice
/usr/local/lib/python3.12/dist-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  warnings.warn(warning.format(ret))
epoch:1-1: loss:5.4934
epoch:1-51: loss:3.1562
epoch:1-101: loss:2.6897
epoch:2-1: loss:2.8529
epoch:2-51: loss:2.2685
epoch:2-101: loss:2.2386
epoch:3-1: loss:1.9368
epoch:3-51: loss:1.5489
epoch:3-101: loss:2.3018
epoch:4-1: loss:1.1722
epoch:4-51: loss:1.6805
epoch:4-101: loss:1.0819
epoch:5-1: loss:1.2872
epoch:5-51: loss:0.9887
epoch:5-101: loss:1.0489
[Saved] /content/ckpt/convadapter/SAM2-UNet-5.pth
epoch:6-1: loss:1.0563
epoch:6-51: loss:1.1082
epoch:6-101: loss:0.9641
epoch:7-1: loss:0.7403
epoch:7-51: loss:0.7652
epoch:7-101: loss:0.7106
epoch:8-1: loss:0.7816
epoch:8-51: loss:0.7468
epoch:8-101: loss:0.6067
epoch:9-1: loss:1.3136
epoch:9-51: loss:0.5625
epoch:9-101: loss:0.6796
epoch:10-1: loss:

In [ ]:
!ls -lh /content/ckpt/convadapter/

total 3.3G
-rw-r--r-- 1 root root 833M May 22 15:46 SAM2-UNet-10.pth
-rw-r--r-- 1 root root 833M May 22 15:52 SAM2-UNet-15.pth
-rw-r--r-- 1 root root 833M May 22 15:57 SAM2-UNet-20.pth
-rw-r--r-- 1 root root 833M May 22 15:40 SAM2-UNet-5.pth


## 8. Test — inference 5 datasets

In [ ]:
%%bash
cd /content/SAM2-UNet
for DATASET in Kvasir CVC-ClinicDB CVC-ColonDB CVC-300 ETIS-LaribPolypDB
do
  echo '========================='
  echo "Testing $DATASET"
  echo '========================='
  python test_convadapter.py \
    --checkpoint /content/ckpt/convadapter/SAM2-UNet-20.pth \
    --test_image_path /content/data/Polyp/TestDataset/$DATASET/images/ \
    --test_gt_path    /content/data/Polyp/TestDataset/$DATASET/masks/ \
    --save_path       /content/preds/convadapter/$DATASET/
done

Testing Kvasir
Saving cju0u82z3cuma0835wlxrnrjv.png
Saving cju15wdt3zla10801odjiw7sy.png
Saving cju16ach3m1da0993r1dq3sn2.png
Saving cju16whaj0e7n0855q7b6cjkm.png
Saving cju17z0qongpa0993de4boim4.png
Saving cju1amqw6p8pw0993d9gc5crl.png
Saving cju1bm8063nmh07996rsjjemq.png
Saving cju1c3218411b08014g9f6gig.png
Saving cju1cbokpuiw70988j4lq1fpi.png
Saving cju1cj3f0qi5n0993ut8f49rj.png
Saving cju1cqc7n4gpy0855jt246k68.png
Saving cju1ddr6p4k5z08780uuuzit2.png
Saving cju1f8w0t65en0799m9oacq0q.png
Saving cju1h89h6xbnx08352k2790o9.png
Saving cju1hp9i2xu8e0988u2dazk7m.png
Saving cju2hfqnmhisa0993gpleeldd.png
Saving cju2hjrqcvi2j0801bx1i6gxg.png
Saving cju2hos57llxm08359g92p6jj.png
Saving cju2hqt33lmra0988fr5ijv8j.png
Saving cju2lberzkdzm09938cl40pog.png
Saving cju2mh8t6p07008350e01tx2a.png
Saving cju2nnqrqzp580855z8mhzgd6.png
Saving cju2np2k9zi3v079992ypxqkn.png
Saving cju2omjpeqj5a0988pjdlb8l1.png
Saving cju2osuru0ki00855txo0n3uu.png
Saving cju2pag1f0s4r0878h52uq83s.png
Saving cju2rga4psq9n098

/content/SAM2-UNet/test_convadapter.py:36: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test_convadapter.py:36: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test_convadapter.py:36: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test_convadapter.py:36: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res = F.upsample(res, size=gt.shape, mode='bilinear', align_corners=False)
/content/SAM2-UNet/test_convadapter.py:36: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  res =

## 9. Evaluate

In [ ]:
%cd /content/SAM2-UNet
!sed -i "s/MSIOU = py_sod_metrics.MSIoU()/MSIOU = py_sod_metrics.MSIoU(with_dynamic=True, with_adaptive=True, with_binary=True)/g" eval.py
print('patched')

/content/SAM2-UNet
patched


In [ ]:
%%bash
cd /content/SAM2-UNet
for DATASET in Kvasir CVC-ClinicDB CVC-ColonDB CVC-300 ETIS-LaribPolypDB
do
  echo '========================='
  echo "Evaluating $DATASET"
  echo '========================='
  python eval.py \
    --dataset_name $DATASET \
    --pred_path /content/preds/convadapter/$DATASET/ \
    --gt_path   /content/data/Polyp/TestDataset/$DATASET/masks/
done

Evaluating Kvasir
[0] Processing cju0u82z3cuma0835wlxrnrjv.png...
[1] Processing cju15wdt3zla10801odjiw7sy.png...
[2] Processing cju16ach3m1da0993r1dq3sn2.png...
[3] Processing cju16whaj0e7n0855q7b6cjkm.png...
[4] Processing cju17z0qongpa0993de4boim4.png...
[5] Processing cju1amqw6p8pw0993d9gc5crl.png...
[6] Processing cju1bm8063nmh07996rsjjemq.png...
[7] Processing cju1c3218411b08014g9f6gig.png...
[8] Processing cju1cbokpuiw70988j4lq1fpi.png...
[9] Processing cju1cj3f0qi5n0993ut8f49rj.png...
[10] Processing cju1cqc7n4gpy0855jt246k68.png...
[11] Processing cju1ddr6p4k5z08780uuuzit2.png...
[12] Processing cju1f8w0t65en0799m9oacq0q.png...
[13] Processing cju1h89h6xbnx08352k2790o9.png...
[14] Processing cju1hp9i2xu8e0988u2dazk7m.png...
[15] Processing cju2hfqnmhisa0993gpleeldd.png...
[16] Processing cju2hjrqcvi2j0801bx1i6gxg.png...
[17] Processing cju2hos57llxm08359g92p6jj.png...
[18] Processing cju2hqt33lmra0988fr5ijv8j.png...
[19] Processing cju2lberzkdzm09938cl40pog.png...
[20] Process

/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 instead!")
/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 instead!")
/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 instead!")
/usr/local/lib/python3.12/dist-packages/py_sod_metrics/sod_metrics.py:35: UserWarning: This class will be removed in the future, please use FmeasureV2 instead!
  warnings.warn("This class will be removed in the future, please use FmeasureV2 ins

## 10. So sánh kết quả — Ablation table

In [ ]:
results = {
    'Baseline          ': {'Kvasir':0.915,'CVC-ClinicDB':0.888,'CVC-ColonDB':0.800,'CVC-300':0.887,'ETIS':0.782},
    '+Dice only        ': {'Kvasir':None, 'CVC-ClinicDB':None, 'CVC-ColonDB':None, 'CVC-300':None, 'ETIS':None},
    '+ConvAdapter only ': {'Kvasir':None, 'CVC-ClinicDB':None, 'CVC-ColonDB':None, 'CVC-300':None, 'ETIS':None},
    '+ConvAdapter+Dice ': {'Kvasir':None, 'CVC-ClinicDB':None, 'CVC-ColonDB':None, 'CVC-300':None, 'ETIS':None},
}
keys=['Kvasir','CVC-ClinicDB','CVC-ColonDB','CVC-300','ETIS']
print(f"{'Model':<22} {'Kvasir':>8} {'ClinicDB':>9} {'ColonDB':>9} {'CVC-300':>8} {'ETIS':>7}")
print('-'*67)
for name,scores in results.items():
    vals=[f"{scores[k]:.3f}" if scores[k] else ' --- ' for k in keys]
    print(f"{name:<22} {'  '.join(f'{v:>7}' for v in vals)}")